# EDA — Telco Customer Churn

Dataset: IBM Telco Customer Churn (~7.000 registros, 20 features, target `Churn`)

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Carregamento e Inspeção Inicial

In [ ]:
df = pd.read_csv('../data/raw/dataset.csv')

print(f'Shape: {df.shape}')
print(f'Duplicatas: {df.duplicated().sum()}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

## 3. TotalCharges — valores não numéricos

`TotalCharges` é lida como `object`. Contém espaços em branco (`" "`) em clientes com `tenure=0`.

In [ ]:
# Linhas com espaço em TotalCharges
espacos = df[df['TotalCharges'].str.strip() == '']
print(f'Linhas com espaço: {len(espacos)}')
espacos[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

In [ ]:
# Conversão: str -> float, espaços viram NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
print(f'NaN após conversão: {df["TotalCharges"].isnull().sum()}')

# Imputação com mediana (conforme spec)
mediana_tc = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(mediana_tc)
print(f'Mediana usada para imputação: {mediana_tc:.2f}')
print(f'NaN após imputação: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Encode target
df['Churn_bin'] = df['Churn'].map({'Yes': 1, 'No': 0})

## 4. Distribuição do Target

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print('Contagem:')
print(churn_counts)
print('\nProporção (%):')
print(churn_pct.round(1))

n_neg = churn_counts['No']
n_pos = churn_counts['Yes']
pos_weight = n_neg / n_pos
print(f'\npos_weight (BCEWithLogitsLoss) = {n_neg}/{n_pos} = {pos_weight:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Distribuição do Target — Churn')
axes[0].set_xticklabels(['Não Churn (No)', 'Churn (Yes)'], rotation=0)
axes[0].set_ylabel('Contagem')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + 0.1, p.get_height() + 30))

churn_pct.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
               colors=['steelblue', 'tomato'], labels=['No', 'Yes'])
axes[1].set_title('Proporção de Classes')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f'\nDataset desbalanceado: 73.5% No vs 26.5% Yes')
print(f'Justifica uso de pos_weight={pos_weight:.2f} no BCEWithLogitsLoss')
print(f'Métrica principal: Recall (FN custa 20x mais que FP) + PR-AUC como técnica')

## 5. Features Numéricas

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[num_cols].describe()

In [ ]:
# Histogramas por classe
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, num_cols):
    for label, color in [('No', 'steelblue'), ('Yes', 'tomato')]:
        ax.hist(df[df['Churn'] == label][col], bins=30, alpha=0.6,
                label=label, color=color, edgecolor='none')
    ax.set_title(f'{col} por Churn')
    ax.set_xlabel(col)
    ax.legend(title='Churn')

plt.suptitle('Distribuição das Features Numéricas por Churn', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por classe
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, num_cols):
    df.boxplot(column=col, by='Churn', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Churn')

plt.suptitle('Boxplots por Churn')
plt.tight_layout()
plt.show()

In [ ]:
# Correlação com target
print('=== Correlação (Pearson) com Churn_bin ===')
for col in num_cols:
    corr = df[col].corr(df['Churn_bin'])
    print(f'  {col:20s}: {corr:+.4f}')

print('\nInterpretação:')
print('  tenure: -0.35  — clientes mais antigos churnam menos (retidos)')
print('  MonthlyCharges: +0.19 — mensalidade alta associada a churn')
print('  TotalCharges: -0.20   — negativo por ser produto de tenure x charges')

## 6. Correlação entre Features (Multicolinearidade)

In [ ]:
corr_num = df[num_cols + ['SeniorCitizen', 'Churn_bin']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_num, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Matriz de Correlação — Features Numéricas')
plt.tight_layout()
plt.show()

tc_tenure = df['tenure'].corr(df['TotalCharges'])
print(f'\ntenure x TotalCharges: {tc_tenure:.4f}')
print(f'ATENÇÃO: correlação {tc_tenure:.3f} — próxima do threshold 0.85')
print('TotalCharges é aproximadamente tenure × MonthlyCharges — redundância conceitual')

In [ ]:
# Verificar pares com correlação > 0.85
corr_matrix = df[num_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(c, r, corr_matrix.loc[r, c])
             for c in upper.columns for r in upper.index
             if pd.notna(upper.loc[r, c]) and upper.loc[r, c] > 0.85]

if high_corr:
    print('Pares com correlação > 0.85:')
    for a, b, v in high_corr:
        print(f'  {a} x {b}: {v:.4f}')
else:
    print('Nenhum par com correlação > 0.85 entre numéricas')

print(f'\ntenure x TotalCharges = {df["tenure"].corr(df["TotalCharges"]):.4f} (abaixo de 0.85 mas candidata a drop)')

## 7. Features Categóricas — Taxa de Churn por Categoria

In [ ]:
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
            'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

for col in cat_cols:
    rates = df.groupby(col)['Churn_bin'].agg(['mean', 'count']).reset_index()
    rates.columns = [col, 'churn_rate', 'count']
    rates['churn_rate'] = (rates['churn_rate'] * 100).round(1)
    rates = rates.sort_values('churn_rate', ascending=False)
    print(rates.to_string(index=False))
    print()

In [ ]:
# Visualizar features mais discriminativas
top_cats = ['Contract', 'InternetService', 'PaymentMethod', 'OnlineSecurity',
            'TechSupport', 'PaperlessBilling', 'gender']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax, col in zip(axes, top_cats):
    rates = df.groupby(col)['Churn_bin'].mean().sort_values(ascending=False)
    bars = ax.bar(range(len(rates)), rates.values * 100,
                  color=['tomato' if v > 0.3 else 'steelblue' for v in rates.values])
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=30, ha='right', fontsize=8)
    ax.set_title(col)
    ax.set_ylabel('Churn %')
    ax.axhline(26.5, color='gray', linestyle='--', linewidth=0.8, label='Média')
    for bar, val in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val*100:.0f}%', ha='center', fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Taxa de Churn por Categoria (linha tracejada = média global 26.5%)', y=1.02)
plt.tight_layout()
plt.show()

## 8. Análise de StreamingTV vs StreamingMovies (possível redundância)

In [ ]:
cross = pd.crosstab(df['StreamingTV'], df['StreamingMovies'], normalize='all') * 100
print('Cross-tab StreamingTV x StreamingMovies (%):')
print(cross.round(1))

tv_rate = df.groupby('StreamingTV')['Churn_bin'].mean()
mov_rate = df.groupby('StreamingMovies')['Churn_bin'].mean()
print('\nChurn rate StreamingTV:', tv_rate.round(3).to_dict())
print('Churn rate StreamingMovies:', mov_rate.round(3).to_dict())
print('\nDistribuições e taxas de churn são quase idênticas — possível redundância')

## 9. Resumo dos Achados e Decisões

In [ ]:
n_neg = (df['Churn'] == 'No').sum()
n_pos = (df['Churn'] == 'Yes').sum()
pos_weight = n_neg / n_pos

print('=' * 60)
print('RESUMO EDA — DECISÕES PARA MODELAGEM')
print('=' * 60)
print()
print(f'Dataset: {df.shape[0]} registros, {df.shape[1]-2} features (sem customerID/Churn_bin)')
print(f'Duplicatas: 0')
print(f'TotalCharges: 11 linhas com espaço → imputadas com mediana ({mediana_tc:.2f})')
print()
print('--- TARGET ---')
print(f'  No Churn: {n_neg} ({n_neg/len(df)*100:.1f}%)')
print(f'  Churn:    {n_pos} ({n_pos/len(df)*100:.1f}%)')
print(f'  Desbalanceamento: {pos_weight:.2f}x')
print()
print('--- MÉTRICAS ---')
print('  Principal: Recall >= 0.75 (restrição de negócio, FN custa 20x)')
print('  Técnica: PR-AUC (mais informativa que AUC-ROC em dados desbalanceados)')
print('  Threshold: otimizar por Expected Profit = TP*1140 - FP*60 - FN*1200')
print()
print('--- POS_WEIGHT ---')
print(f'  pos_weight = {n_neg}/{n_pos} = {pos_weight:.4f}')
print()
print('--- CORRELAÇÕES CRÍTICAS ---')
print(f'  tenure x TotalCharges: 0.826 (abaixo de 0.85, mas redundância conceitual)')
print(f'  StreamingTV x StreamingMovies: distribuições quase idênticas')
print()
print('--- FEATURES CANDIDATAS A DROP ---')
print('  gender: churn 26.9% vs 26.2% — sem poder discriminativo')
print('  TotalCharges: alta correlação com tenure (0.826) — avaliar no pipeline')
print('  PhoneService: churn 26.7% vs 24.9% — sinal fraco')
print()
print('--- FEATURES MAIS PREDITIVAS ---')
print('  Contract (Month-to-month: 42.7% vs Two year: 2.8%)')
print('  InternetService (Fiber optic: 41.9% vs No: 7.4%)')
print('  PaymentMethod (Electronic check: 45.3%)')
print('  OnlineSecurity / TechSupport (~42% sem vs ~15% com serviço)')
print('  tenure (correlação -0.35 com churn)')